# Exploratory Analysis — Texas Afghan Support Program

This notebook explores synthetic ZIP-code-level program data across Austin, Dallas, Fort Worth, Houston, and San Antonio.

The analysis focuses on four questions:

1. Where are households, participants, and rental assistance concentrated?
2. How did enrollment, rental assistance, and Financial Wellness Course attendance change from July through September?
3. Which ZIP codes experienced the largest month-to-month increases or declines?
4. How are program volume, assistance, and course attendance related?

**Scope:** descriptive exploratory analysis. These aggregate data support comparisons, distributions, trends, and associations, but not individual-level or causal conclusions.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data" / "asp_zip_metrics.csv"

df = pd.read_csv(DATA, dtype={"zip_code": str})

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

df.head()

,zip_code,city,total_households,total_participants,ra_july,ra_august,ra_september,ra_total,fwc_july,fwc_august,fwc_september,fwc_total,enrolled_july_cumulative,enrolled_august_cumulative,enrolled_september_cumulative,top10_ra_city,latitude,longitude,ra_per_household,participants_per_household,ra_aug_vs_july_pct,ra_sep_vs_aug_pct
0,78610,Austin,1,1,0.00,"2,351.00","2,534.00","4,885.00",1,1,0,1,0,1,1,False,30.08,-97.84,"4,885.00",1.00,NaN,7.78
1,78621,Austin,1,2,0.00,0.00,"1,772.00","1,772.00",1,2,0,2,0,0,1,False,30.34,-97.36,"1,772.00",2.00,NaN,NaN
2,78626,Austin,3,4,0.00,"1,604.00","5,147.00","6,751.00",2,4,0,4,0,1,3,False,30.66,-97.62,"2,250.33",1.33,NaN,220.89
3,78628,Austin,3,5,"3,709.00","5,388.00","5,319.00","14,416.00",4,4,1,5,2,3,3,False,30.66,-97.78,"4,805.33",1.67,45.27,-1.28
4,78640,Austin,3,4,"2,212.00","2,121.00","2,332.00","6,665.00",4,4,0,4,2,3,3,False,29.99,-97.82,"2,221.67",1.33,-4.11,9.95


## 1. Data overview and quality checks

In [2]:
quality = pd.Series({
    "Rows": len(df),
    "Unique ZIP codes": df["zip_code"].nunique(),
    "Metro areas": df["city"].nunique(),
    "Missing city": int(df["city"].isna().sum()),
    "Missing total rental assistance": int(df["ra_total"].isna().sum()),
    "Missing FWC total": int(df["fwc_total"].isna().sum()),
})

quality

Rows                               192
Unique ZIP codes                   192
Metro areas                          5
Missing city                         0
Missing total rental assistance      0
Missing FWC total                    0
dtype: int64

In [3]:
monthly_ra_sum = df["ra_july"] + df["ra_august"] + df["ra_september"]
ra_reconciliation_error = (monthly_ra_sum - df["ra_total"]).abs()

pd.Series({
    "Maximum RA reconciliation error": ra_reconciliation_error.max(),
    "ZIP codes with reconciliation error > $0.01": int((ra_reconciliation_error > 0.01).sum()),
})

Maximum RA reconciliation error               0.00
ZIP codes with reconciliation error > $0.01   0.00
dtype: float64

The rental-assistance total reconciles to the sum of July, August, and September at the cent level. That internal-consistency check supports using the monthly and total measures together in the analysis.

## 2. Descriptive statistics

In [4]:
overall = pd.Series({
    "ZIP codes": len(df),
    "Households": df["total_households"].sum(),
    "Participants": df["total_participants"].sum(),
    "Total rental assistance": df["ra_total"].sum(),
    "Median ZIP rental assistance": df["ra_total"].median(),
    "Total unique FWC attendees": df["fwc_total"].sum(),
})

overall

ZIP codes                             192.00
Households                          3,500.00
Participants                        6,008.00
Total rental assistance        11,744,050.00
Median ZIP rental assistance       13,656.00
Total unique FWC attendees          6,471.00
dtype: float64

In [5]:
df[
    [
        "total_households",
        "total_participants",
        "ra_total",
        "fwc_total",
        "ra_per_household",
        "participants_per_household",
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
total_households,192.00,18.23,52.79,1.00,1.00,3.00,12.00,378.00
total_participants,192.00,31.29,93.08,1.00,2.00,5.50,21.00,665.00
ra_total,192.00,"61,166.93","168,939.70",587.00,"4,870.50","13,656.00","42,031.25","1,180,664.00"
fwc_total,192.00,33.70,100.93,1.00,2.00,6.00,21.25,763.00
ra_per_household,192.00,"3,735.32","1,332.06",587.00,"2,910.03","3,615.00","4,364.69","10,366.00"
participants_per_household,192.00,1.64,0.57,1.00,1.06,1.58,2.00,4.00


ZIP-level rental assistance is strongly right-skewed: a relatively small number of ZIP codes account for substantially larger totals than the typical ZIP. This makes medians, quantiles, and concentration measures more informative than averages alone.

## 3. Metro-area comparison

In [6]:
city = (
    df.groupby("city", as_index=False)
      .agg(
          zip_codes=("zip_code", "nunique"),
          households=("total_households", "sum"),
          participants=("total_participants", "sum"),
          rental_assistance=("ra_total", "sum"),
          fwc_attendees=("fwc_total", "sum"),
      )
)

city["ra_per_household"] = city["rental_assistance"] / city["households"]
city["participants_per_household"] = city["participants"] / city["households"]

city.sort_values("rental_assistance", ascending=False)

,city,zip_codes,households,participants,rental_assistance,fwc_attendees,ra_per_household,participants_per_household
3,Houston,44,1327,2362,"4,213,271.00",2745,"3,175.03",1.78
4,San Antonio,27,745,1254,"2,436,173.00",1423,"3,270.03",1.68
1,Dallas,47,652,1186,"2,231,722.00",1074,"3,422.89",1.82
2,Fort Worth,37,417,636,"1,450,395.00",639,"3,478.17",1.53
0,Austin,37,359,570,"1,412,489.00",590,"3,934.51",1.59


In [7]:
plot_city = city.sort_values("rental_assistance")

plt.figure(figsize=(8, 5))
plt.barh(plot_city["city"], plot_city["rental_assistance"])
plt.xlabel("Rental assistance ($)")
plt.ylabel("Metro area")
plt.title("Rental Assistance by Metro Area")
plt.tight_layout()
plt.show()

## 4. Distribution and concentration

In [8]:
plt.figure(figsize=(8, 5))
plt.hist(df["ra_total"], bins=30)
plt.xlabel("Total rental assistance per ZIP ($)")
plt.ylabel("Number of ZIP codes")
plt.title("Distribution of ZIP-Level Rental Assistance")
plt.tight_layout()
plt.show()

In [9]:
top10 = (
    df.nlargest(10, "ra_total")[
        ["zip_code", "city", "total_households", "total_participants", "ra_total"]
    ]
    .copy()
)

top10["share_of_total_ra"] = top10["ra_total"] / df["ra_total"].sum()
top10

,zip_code,city,total_households,total_participants,ra_total,share_of_total_ra
176,78229,San Antonio,378,641,"1,180,664.00",0.10
76,75243,Dallas,326,598,"1,123,453.00",0.10
135,77063,Houston,358,665,"1,116,607.00",0.10
139,77074,Houston,273,486,"805,976.00",0.07
180,78240,San Antonio,205,320,"709,215.00",0.06
106,76119,Fort Worth,137,220,"458,221.00",0.04
32,78753,Austin,95,159,"380,404.00",0.03
143,77081,Houston,137,217,"349,108.00",0.03
72,75231,Dallas,112,187,"326,219.00",0.03
130,77042,Houston,93,190,"312,010.00",0.03


In [10]:
top10_share = top10["ra_total"].sum() / df["ra_total"].sum()
print(f"Top 10 ZIP codes account for {top10_share:.1%} of total rental assistance.")

Top 10 ZIP codes account for 57.6% of total rental assistance.


This concentration measure complements the choropleth map: the map shows where high-volume ZIP codes are located, while the concentration statistic quantifies how much of the total assistance they represent.

## 5. Monthly program dynamics

### 5.1 Enrollment

In [11]:
enrollment_monthly = pd.Series({
    "July": df["enrolled_july_cumulative"].sum(),
    "August": df["enrolled_august_cumulative"].sum(),
    "September": df["enrolled_september_cumulative"].sum(),
})

enrollment_mom = enrollment_monthly.pct_change() * 100

pd.DataFrame({
    "Cumulative households enrolled": enrollment_monthly,
    "MoM change (%)": enrollment_mom,
})

,Cumulative households enrolled,MoM change (%)
July,2352,NaN
August,3449,46.64
September,3500,1.48


In [12]:
plt.figure(figsize=(7, 4))
plt.plot(enrollment_monthly.index, enrollment_monthly.values, marker="o")
plt.ylabel("Cumulative households enrolled")
plt.title("Cumulative Enrollment by Month")
plt.tight_layout()
plt.show()

Because enrollment is cumulative, the August and September values should be interpreted as the total enrolled by that point in time rather than as standalone monthly counts.

### 5.2 Rental assistance

In [13]:
ra_monthly = pd.Series({
    "July": df["ra_july"].sum(),
    "August": df["ra_august"].sum(),
    "September": df["ra_september"].sum(),
})

ra_mom = ra_monthly.pct_change() * 100

pd.DataFrame({
    "Rental assistance": ra_monthly,
    "MoM change (%)": ra_mom,
})

,Rental assistance,MoM change (%)
July,"3,040,721.00",NaN
August,"4,386,600.00",44.26
September,"4,316,729.00",-1.59


In [14]:
plt.figure(figsize=(7, 4))
plt.plot(ra_monthly.index, ra_monthly.values, marker="o")
plt.ylabel("Rental assistance ($)")
plt.title("Monthly Rental Assistance")
plt.tight_layout()
plt.show()

### 5.3 Financial Wellness Course attendance

In [15]:
fwc_monthly = pd.Series({
    "July": df["fwc_july"].sum(),
    "August": df["fwc_august"].sum(),
    "September": df["fwc_september"].sum(),
})

fwc_mom = fwc_monthly.pct_change() * 100

pd.DataFrame({
    "Monthly FWC attendance": fwc_monthly,
    "MoM change (%)": fwc_mom,
})

,Monthly FWC attendance,MoM change (%)
July,5025,NaN
August,5748,14.39
September,1604,-72.09


In [16]:
plt.figure(figsize=(7, 4))
plt.plot(fwc_monthly.index, fwc_monthly.values, marker="o")
plt.ylabel("Monthly attendees")
plt.title("Financial Wellness Course Attendance by Month")
plt.tight_layout()
plt.show()

Monthly FWC attendance counts unique attendees within each month. `fwc_total` is a different measure: it represents unique attendees across the entire July–September period, so the monthly values should not be summed to reconstruct the total.

## 6. Metro-area monthly trends

In [17]:
city_monthly = (
    df.groupby("city")
      .agg(
          enrollment_july=("enrolled_july_cumulative", "sum"),
          enrollment_august=("enrolled_august_cumulative", "sum"),
          enrollment_september=("enrolled_september_cumulative", "sum"),
          ra_july=("ra_july", "sum"),
          ra_august=("ra_august", "sum"),
          ra_september=("ra_september", "sum"),
          fwc_july=("fwc_july", "sum"),
          fwc_august=("fwc_august", "sum"),
          fwc_september=("fwc_september", "sum"),
      )
)

city_monthly

,enrollment_july,enrollment_august,enrollment_september,ra_july,ra_august,ra_september,fwc_july,fwc_august,fwc_september
city,,,,,,,,,
Austin,282,347,359,"399,256.00","495,057.00","518,176.00",575,521,34
Dallas,349,636,652,"508,980.00","882,194.00","840,548.00",716,886,40
Fort Worth,244,417,417,"333,892.00","558,691.00","557,812.00",408,620,125
Houston,876,1314,1327,"1,075,114.00","1,581,052.00","1,557,105.00",2030,2399,704
San Antonio,601,735,745,"723,479.00","869,606.00","843,088.00",1296,1322,701


In [18]:
ra_city_plot = city_monthly[["ra_july", "ra_august", "ra_september"]].T
ra_city_plot.index = ["July", "August", "September"]

plt.figure(figsize=(9, 5))
for city_name in ra_city_plot.columns:
    plt.plot(ra_city_plot.index, ra_city_plot[city_name], marker="o", label=city_name)

plt.ylabel("Rental assistance ($)")
plt.title("Monthly Rental Assistance by Metro Area")
plt.legend()
plt.tight_layout()
plt.show()

In [19]:
fwc_city_plot = city_monthly[["fwc_july", "fwc_august", "fwc_september"]].T
fwc_city_plot.index = ["July", "August", "September"]

plt.figure(figsize=(9, 5))
for city_name in fwc_city_plot.columns:
    plt.plot(fwc_city_plot.index, fwc_city_plot[city_name], marker="o", label=city_name)

plt.ylabel("Monthly attendees")
plt.title("Monthly FWC Attendance by Metro Area")
plt.legend()
plt.tight_layout()
plt.show()

## 7. ZIP-level growth and decline

In [20]:
df["enrollment_aug_mom_pct"] = np.where(
    df["enrolled_july_cumulative"] > 0,
    (df["enrolled_august_cumulative"] - df["enrolled_july_cumulative"])
    / df["enrolled_july_cumulative"] * 100,
    np.nan,
)

df["enrollment_sep_mom_pct"] = np.where(
    df["enrolled_august_cumulative"] > 0,
    (df["enrolled_september_cumulative"] - df["enrolled_august_cumulative"])
    / df["enrolled_august_cumulative"] * 100,
    np.nan,
)

df["fwc_aug_mom_pct"] = np.where(
    df["fwc_july"] > 0,
    (df["fwc_august"] - df["fwc_july"]) / df["fwc_july"] * 100,
    np.nan,
)

df["fwc_sep_mom_pct"] = np.where(
    df["fwc_august"] > 0,
    (df["fwc_september"] - df["fwc_august"]) / df["fwc_august"] * 100,
    np.nan,
)

df[
    [
        "zip_code",
        "city",
        "enrollment_aug_mom_pct",
        "enrollment_sep_mom_pct",
        "ra_aug_vs_july_pct",
        "ra_sep_vs_aug_pct",
        "fwc_aug_mom_pct",
        "fwc_sep_mom_pct",
    ]
].head()

,zip_code,city,enrollment_aug_mom_pct,enrollment_sep_mom_pct,ra_aug_vs_july_pct,ra_sep_vs_aug_pct,fwc_aug_mom_pct,fwc_sep_mom_pct
0,78610,Austin,NaN,0.00,NaN,7.78,0.00,-100.00
1,78621,Austin,NaN,NaN,NaN,NaN,100.00,-100.00
2,78626,Austin,NaN,200.00,NaN,220.89,100.00,-100.00
3,78628,Austin,50.00,0.00,45.27,-1.28,0.00,-75.00
4,78640,Austin,50.00,0.00,-4.11,9.95,0.00,-100.00


### Largest August-to-September RA increases

In [21]:
df.nlargest(10, "ra_sep_vs_aug_pct")[
    ["zip_code", "city", "ra_august", "ra_september", "ra_sep_vs_aug_pct"]
]

,zip_code,city,ra_august,ra_september,ra_sep_vs_aug_pct
2,78626,Austin,"1,604.00","5,147.00",220.89
53,75061,Dallas,"1,663.00","3,529.00",112.21
62,75080,Dallas,"1,576.00","2,932.00",86.04
48,75043,Dallas,"4,635.00","8,122.00",75.23
82,75407,Dallas,"4,027.00","5,884.00",46.11
60,75074,Dallas,"5,286.00","7,091.00",34.15
184,78251,San Antonio,"4,380.00","5,636.00",28.68
98,76108,Fort Worth,"13,438.00","15,112.00",12.46
38,75006,Dallas,"2,070.00","2,319.00",12.03
150,77095,Houston,"1,068.00","1,196.00",11.99


### Largest August-to-September RA declines

In [22]:
df.nsmallest(10, "ra_sep_vs_aug_pct")[
    ["zip_code", "city", "ra_august", "ra_september", "ra_sep_vs_aug_pct"]
]

,zip_code,city,ra_august,ra_september,ra_sep_vs_aug_pct
123,77008,Houston,"1,106.00",0.00,-100.00
158,77459,Houston,"2,135.00",0.00,-100.00
71,75230,Dallas,"1,965.00",924.00,-52.98
49,75044,Dallas,"24,048.00","17,729.00",-26.28
170,78216,San Antonio,"3,471.00","2,569.00",-25.99
37,75002,Dallas,"9,842.00","7,439.00",-24.42
122,77007,Houston,"20,389.00","17,370.00",-14.81
103,76115,Fort Worth,"22,233.00","19,304.00",-13.17
148,77092,Houston,"22,837.00","19,879.00",-12.95
130,77042,Houston,"116,804.00","103,347.00",-11.52


### Largest August-to-September enrollment increases

In [23]:
df.nlargest(10, "enrollment_sep_mom_pct")[
    [
        "zip_code",
        "city",
        "enrolled_august_cumulative",
        "enrolled_september_cumulative",
        "enrollment_sep_mom_pct",
    ]
]

,zip_code,city,enrolled_august_cumulative,enrolled_september_cumulative,enrollment_sep_mom_pct
2,78626,Austin,1,3,200.00
53,75061,Dallas,1,2,100.00
62,75080,Dallas,1,2,100.00
48,75043,Dallas,3,5,66.67
82,75407,Dallas,2,3,50.00
60,75074,Dallas,3,4,33.33
162,77479,Houston,20,23,15.00
40,75013,Dallas,7,8,14.29
175,78228,San Antonio,10,11,10.00
25,78741,Austin,29,31,6.90


Percentage growth can be large when the prior-month base is small. The rankings should therefore be read alongside the underlying counts rather than interpreted from percentages alone.

## 8. Relationships among program measures

In [24]:
corr = df[
    [
        "total_households",
        "total_participants",
        "ra_total",
        "fwc_total",
        "ra_per_household",
    ]
].corr()

corr

,total_households,total_participants,ra_total,fwc_total,ra_per_household
total_households,1.00,1.00,1.00,0.99,-0.10
total_participants,1.00,1.00,0.99,0.99,-0.09
ra_total,1.00,0.99,1.00,0.99,-0.08
fwc_total,0.99,0.99,0.99,1.00,-0.10
ra_per_household,-0.10,-0.09,-0.08,-0.10,1.00


In [25]:
plt.figure(figsize=(7, 5))
plt.scatter(df["total_households"], df["ra_total"], alpha=0.65)
plt.xlabel("Households")
plt.ylabel("Rental assistance ($)")
plt.title("Households and Rental Assistance by ZIP Code")
plt.tight_layout()
plt.show()

In [26]:
plt.figure(figsize=(7, 5))
plt.scatter(df["total_participants"], df["fwc_total"], alpha=0.65)
plt.xlabel("Participants")
plt.ylabel("Unique FWC attendees")
plt.title("Program Participation and FWC Attendance by ZIP Code")
plt.tight_layout()
plt.show()

The strongest relationships in the dataset are expected to be volume-driven: ZIP codes with more participating households generally have more participants, more rental assistance, and more course attendees. These are descriptive associations and should not be interpreted as causal effects.

## 9. Normalized measures

In [27]:
normalized = (
    df.groupby("city", as_index=False)
      .agg(
          households=("total_households", "sum"),
          participants=("total_participants", "sum"),
          rental_assistance=("ra_total", "sum"),
          fwc_attendees=("fwc_total", "sum"),
      )
)

normalized["ra_per_household"] = normalized["rental_assistance"] / normalized["households"]
normalized["fwc_attendees_per_100_participants"] = (
    normalized["fwc_attendees"] / normalized["participants"] * 100
)

normalized.sort_values("ra_per_household", ascending=False)

,city,households,participants,rental_assistance,fwc_attendees,ra_per_household,fwc_attendees_per_100_participants
0,Austin,359,570,"1,412,489.00",590,"3,934.51",103.51
2,Fort Worth,417,636,"1,450,395.00",639,"3,478.17",100.47
1,Dallas,652,1186,"2,231,722.00",1074,"3,422.89",90.56
4,San Antonio,745,1254,"2,436,173.00",1423,"3,270.03",113.48
3,Houston,1327,2362,"4,213,271.00",2745,"3,175.03",116.22


Normalizing by program size changes the question from “where is the most activity?” to “how much activity is occurring relative to the number of participating households or participants?” This helps distinguish scale from intensity.

## 10. Key findings

In [28]:
largest_city = city.loc[city["rental_assistance"].idxmax()]
largest_zip = df.loc[df["ra_total"].idxmax()]

summary = pd.Series({
    "Metro area with highest total RA": largest_city["city"],
    "Highest-RA ZIP": largest_zip["zip_code"],
    "Highest-RA ZIP metro": largest_zip["city"],
    "Highest ZIP RA": largest_zip["ra_total"],
    "Top-10 ZIP share of total RA": top10_share,
    "August statewide RA MoM change (%)": ra_mom["August"],
    "September statewide RA MoM change (%)": ra_mom["September"],
    "August statewide FWC MoM change (%)": fwc_mom["August"],
    "September statewide FWC MoM change (%)": fwc_mom["September"],
})

summary

Metro area with highest total RA              Houston
Highest-RA ZIP                                  78229
Highest-RA ZIP metro                      San Antonio
Highest ZIP RA                           1,180,664.00
Top-10 ZIP share of total RA                     0.58
August statewide RA MoM change (%)              44.26
September statewide RA MoM change (%)           -1.59
August statewide FWC MoM change (%)             14.39
September statewide FWC MoM change (%)         -72.09
dtype: object

### Interpretation

The project shows a clear geographic concentration of program activity. Houston carries the largest overall rental-assistance volume, while a relatively small number of ZIP codes account for a substantial share of total assistance.

The month-to-month analysis adds context that a single cumulative map cannot show on its own. Enrollment, rental assistance, and Financial Wellness attendance follow different temporal patterns, and ZIP-level percentage changes vary widely.

For portfolio purposes, the key analytical lesson is that **scale, change, and intensity are different concepts**:

- total dollars identify high-volume areas,
- MoM percentages identify changing areas,
- per-household and per-participant measures normalize for program size,
- maps reveal the geographic pattern behind those statistics.


## 11. Analytical limitations

This project uses synthetic aggregate ZIP-code-level data. Several limitations are important:

- ZIP-level aggregates cannot support participant-level inference.
- Correlations among program measures do not establish causation.
- Month-over-month percentages can be volatile when the prior-month count is small.
- Cumulative enrollment is conceptually different from monthly RA and monthly FWC attendance.
- Monthly FWC values count unique attendees within each month, while `fwc_total` counts unique attendees across the full July–September period.

The analysis is therefore best interpreted as descriptive program-monitoring and geospatial analytics.
